# Detailed Checklist — GDP Growth Modeling & Similar Panel-Data Projects

Markdown checklists — click into a cell and toggle `[ ]` to `[x]`, or just track progress visually. Organized in four parts:

1. **This project's step-by-step checklist**
2. **Transferable checklist** for any similar economic/panel-data regression project
3. **Common pitfalls checklist** (including accounting-format and structural-missingness traps)
4. **Definition-of-done checklist** for the whole project


## Part 1 — This Project's Step-by-Step Checklist

### Step 1 — Load & Inspect
- [ ] Import pandas, numpy, matplotlib, seaborn
- [ ] Load `economy_indicators.csv` into `df`
- [ ] Print `df.shape`, `df.head()`, `df.info()`
- [ ] Run `df.describe(include='all').T`
- [ ] Eyeball `gdp_growth_rate_pct` and `inflation_rate_pct` for a negative-number formatting convention beyond a plain minus sign


### Step 2 — Data Quality Audit
- [ ] Build a dtype/nunique/missing audit table
- [ ] Check `df.duplicated().sum()`; drop confirmed duplicates
- [ ] Count distinct values of `country`; investigate whitespace and casing inconsistencies
- [ ] Clean `country` with strip + whitespace-collapse + title-case
- [ ] Find every column using the `".."` sentinel
- [ ] Cross-tabulate `disaster_damage_pct_gdp`'s `".."` rows against `natural_disaster_event` to confirm the structural-missingness pattern


### Step 3 — Accounting-Style Percentage Parsing (Including the Target)
- [ ] Write a parentheses-aware percentage parser
- [ ] Apply it to every percentage column
- [ ] **Explicitly confirm you applied it to `gdp_growth_rate_pct` (the target) and `imf_next_year_growth_forecast_pct`, not just the "obvious" feature columns**
- [ ] Spot-check a few known-negative rows against the parsed output


### Step 4 — Two Kinds of Missing Values
- [ ] `fdi_inflow_pct_gdp`, `education_spending_pct_gdp`, `tourism_arrivals_millions`: flag + median impute (random reporting gaps)
- [ ] `disaster_damage_pct_gdp`: impute with 0, no flag (structural absence) — written justification included
- [ ] Clean `population` and `gdp_per_capita_usd` (commas, `$`, unit suffixes)
- [ ] Convert `imf_program_active`/`natural_disaster_event` to 0/1


### Step 5 — Ordinal Encoding & the Hand-Crafted Interaction
- [ ] Map `income_group` to an ordinal score
- [ ] Create `disaster_debt_interaction`
- [ ] Written note on why a tree model doesn't strictly need this hand-crafted feature while a linear model does


### Step 6 — Advanced EDA & the Forecast-Leakage Audit
- [ ] Target distribution plotted; skewness checked
- [ ] Correlation heatmap built **including** the IMF forecast column
- [ ] Explicit correlation value computed
- [ ] Written reasoning: why exclude a column with a *moderate* (not extreme) correlation, given this project's specific "independent, scenario-responsive" goal
- [ ] VIF computed on the retained feature set
- [ ] Boxplots by `natural_disaster_event` and `currency_regime`
- [ ] Interaction-feature correlation compared against its two source columns' individual correlations


### Step 7 — Leakage-Safe Encoding
- [ ] Ordinal column's original text version excluded from further encoding
- [ ] Remaining categoricals bucketed by cardinality
- [ ] Train/test split performed before any target-encoding statistic
- [ ] Target-encoding maps fit on train only, with an unseen-category fallback
- [ ] Zero `NaN`s confirmed in `X_train`/`X_test`


### Step 8 — Baseline & Linear Models
- [ ] Mean-predictor baseline scored
- [ ] `LinearRegression` scored **without** the interaction term
- [ ] `LinearRegression` scored **with** the interaction term; improvement quantified
- [ ] `Ridge` scored


### Step 9 — Tree-Ensemble Models
- [ ] `RandomForestRegressor` and `GradientBoostingRegressor` (default) scored
- [ ] All models collected into one comparison table
- [ ] Written discussion: why might a sparse interaction be harder for a default tree ensemble to find than for an explicitly-engineered linear feature?


### Step 10 — Cross-Validation & Tuning
- [ ] 5-fold CV MAE computed for the leading candidate
- [ ] Hyperparameter distribution defined; `RandomizedSearchCV` run
- [ ] Tuned model refit and evaluated on the held-out test set


### Step 11 — Evaluation & Diagnostics
- [ ] MAE, RMSE, R² reported; MAPE reported with a near-zero-growth caveat
- [ ] Predicted-vs-actual scatterplot
- [ ] Residuals compared for disaster vs. non-disaster quarters specifically


### Step 12 — Feature Importance
- [ ] Coefficient or impurity-based top-10 plotted
- [ ] Permutation importance top-10 plotted
- [ ] Checked where `disaster_debt_interaction` ranks in each method


### Step 13 — Policy Scenario Simulation
- [ ] A baseline row selected (Costa Rica, per this project's framing)
- [ ] Commodity price shock scenario simulated
- [ ] Tourism revenue shock scenario simulated
- [ ] Disaster scenario compared at two different debt levels
- [ ] Written note on what this offers beyond a single fixed forecast, and a caution against extrapolating too far from the training data's range


### Step 14 — Persistence & Inference
- [ ] Final model saved with `joblib.dump`
- [ ] Encoding maps, ordinal map, and column order saved alongside the model
- [ ] `predict_growth()` function written, reproducing every training-time transform
- [ ] Function tested on 2–3 made-up country-quarters


### Step 15 — Conclusions
- [ ] Final model choice stated, including whether the interaction term mattered
- [ ] Expected error stated in percentage points
- [ ] Top 3–5 growth drivers named
- [ ] One scenario-simulation finding highlighted
- [ ] At least one limitation and one next step named


## Part 2 — Transferable Checklist for Economic/Panel-Data Regression Projects

### Phase 1 — Understand the Problem
- [ ] Target and any forecast/quote-style comparison column identified
- [ ] Project goal stated precisely (predict the forecast vs. build an independent, input-responsive model)
- [ ] Whether a scenario-simulation deliverable is needed considered up front (it shapes which columns must be excluded even at moderate correlation)


### Phase 2 — Data Acquisition & Audit
- [ ] Shape, dtypes, nulls, duplicates all checked
- [ ] Every percentage/currency column checked for accounting-style negative formatting, **including the target**
- [ ] Entity-identifier columns (country/company names) checked for whitespace/casing fragmentation
- [ ] Every sentinel-missing-value convention identified (`".."`, `"N/A"`, etc.)


### Phase 3 — Feature Engineering
- [ ] Accounting-style percentages parsed correctly (parentheses = negative)
- [ ] For each missing-value column: cross-tabulated against a suspected condition column to distinguish structural absence from random reporting gaps
- [ ] Structural-missing columns imputed with a domain-deduced constant, no flag
- [ ] Random-reporting-gap columns imputed with a statistical estimate, with a flag
- [ ] Naturally-ordered categories ordinal-encoded
- [ ] Any sparse-but-plausible interaction hand-engineered, with a written domain justification


### Phase 4 — The Forecast/Quote Leakage Audit
- [ ] Every forecast/quote-style column identified
- [ ] Correlation with target computed (don't assume it must be extreme to matter)
- [ ] For each: reasoned about what it represents, not just its correlation strength, before deciding inclusion/exclusion
- [ ] VIF checked on the retained feature set


### Phase 5 — Encoding & Splitting
- [ ] Ordinal columns' original text versions excluded from further (one-hot/target) encoding
- [ ] Remaining nominal categoricals partitioned by cardinality
- [ ] Split performed before any target-dependent computation
- [ ] Target encoding fit on train only, with an unseen-category fallback


### Phase 6 — Modeling
- [ ] Linear model compared with AND without any hand-engineered interaction
- [ ] At least one tree ensemble compared alongside
- [ ] No model family assumed to win in advance
- [ ] If an interaction was engineered: its contribution quantified, not just asserted


### Phase 7 — Evaluation, Interpretation & Scenario Simulation
- [ ] Metrics matched to the target's scale; near-zero-value MAPE instability flagged if relevant
- [ ] Feature importance computed via two independent methods
- [ ] If scenario simulation is part of the deliverable: each scenario checked against the training data's range before trusting it
- [ ] Results reported with appropriate caution about model simplifications


### Phase 8 — Communicate & Persist
- [ ] Findings summarized in plain language, with confidence caveats
- [ ] Model + preprocessing artifacts (including ordinal maps, interaction-recompute logic) persisted together
- [ ] Limitations and next steps explicitly named


## Part 3 — Common Pitfalls Checklist

- [ ] **Accounting-format blindness** — treating `"(1.7%)"` as unparseable/missing instead of recognizing the parentheses-negative convention
- [ ] **Forgetting the target itself needs cleaning** — applying careful parsing to features but assuming the target column is already clean
- [ ] **Conflating structural absence with random missingness** — imputing a deducible zero (or other domain constant) with a statistical median instead
- [ ] **Skipping the missingness cross-tabulation** — guessing at a structural-missingness pattern instead of verifying it against the suspected condition column
- [ ] **Entity-identifier fragmentation** — leaving whitespace/casing variants of the same country/company split into separate categories
- [ ] **Correlation-threshold tunnel vision** — excluding or including a forecast/quote column based purely on a correlation cutoff, without reasoning about what it represents
- [ ] **Assuming a sparse interaction will be found automatically** — skipping explicit feature engineering because "trees can learn interactions," even when the relevant subset of data is too small for that to reliably happen
- [ ] **Scenario-simulation overreach** — trusting a "what if" prediction for an input combination far outside anything the training data contained
- [ ] **MAPE misuse** — reporting MAPE on a target that can sit near zero without flagging the instability
- [ ] **Model without its preprocessing** — persisting a model without the encoding/ordinal maps and interaction-recompute logic needed to use it on new data


## Part 4 — Definition-of-Done Checklist for the Whole Project

- [ ] Every raw column is either numeric, properly encoded, or intentionally dropped with a stated reason
- [ ] The accounting-format percentage parser was applied everywhere it was needed, including the target
- [ ] Every missing-value column has an explicit, justified classification (structural vs. random) and matching imputation strategy
- [ ] No `NaN` values remain anywhere in the final training/test feature matrices
- [ ] The forecast/quote column has an explicit, written inclusion/exclusion decision with reasoning beyond correlation alone
- [ ] A linear model with a hand-engineered interaction, a plain linear model, and at least one tree ensemble were all compared
- [ ] Final model selected with a written justification
- [ ] Cross-validated performance estimate reported alongside single-split test metrics
- [ ] Two independent feature-importance methods agree on (most of) the top drivers
- [ ] If scenario simulation was performed: results were checked against the training data's range and reported with appropriate caution
- [ ] Model and preprocessing artifacts are persisted and reloadable
- [ ] A plain-language summary exists, including explicit caveats about synthetic/limited data
